# NER offset test

Run with the **contracts-engine** interpreter (py3.12). spaCy runs in-kernel; LexNLP runs via the py3.8 container. Edit the config cell, then run top to bottom.

In [1]:
from pathlib import Path
import subprocess, tempfile
import pandas as pd
import pyarrow.dataset as pads

# --- config: edit these -------------------------------------------------
INPUTS       = ["/Users/matthiasuckert/RProjects/Projects/material-contracts/0_sample/2021-1"]
ID_COL       = "DocID"
TEXT_COL     = "TextRaw"            # default; point at a rendered column if you make one
MODEL        = "en_core_web_sm"     # or _md / _lg / _trf
N_DOCS       = 25
LEXNLP_IMAGE = "contracts-lexnlp"
# ------------------------------------------------------------------------

def resolve_inputs(paths):
    files = []
    for p in paths:
        pth = Path(p)
        files += sorted(str(f) for f in pth.rglob("*.parquet")) if pth.is_dir() else [str(pth)]
    return files

df = (pads.dataset(resolve_inputs(INPUTS), format="parquet")
          .head(N_DOCS, columns=[ID_COL, TEXT_COL])
          .to_pandas())
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
text_by_id = dict(zip(df[ID_COL], df[TEXT_COL]))
df.shape

(25, 2)

## spaCy (in-kernel)

In [3]:
import spacy
LABEL_MAP = {"ORG": "ORG", "PERSON": "PERSON", "GPE": "GPE", "LOC": "GPE"}

nlp = spacy.load(MODEL)
for pipe in ("parser", "lemmatizer", "tagger", "attribute_ruler", "morphologizer"):
    if pipe in nlp.pipe_names:
        nlp.disable_pipe(pipe)
nlp.max_length = 5_000_000

rows = []
for docid, doc in zip(df[ID_COL], nlp.pipe(df[TEXT_COL].tolist(), batch_size=32)):
    for ent in doc.ents:
        rows.append({"DocID": docid, "Start": ent.start_char, "Stop": ent.end_char,
                     "Span": ent.text, "Label": LABEL_MAP.get(ent.label_, ent.label_),
                     "LabelRaw": ent.label_, "Engine": "spacy"})
spacy_df = pd.DataFrame(rows)
print(len(spacy_df), "spaCy candidates")
spacy_df.head(10)

/Users/matthiasuckert/RProjects/Projects/material-contracts/contracts-engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


17840 spaCy candidates


,DocID,Start,Stop,Span,Label,LabelRaw,Engine
0,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,9,10,4,CARDINAL,CARDINAL,spacy
1,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,44,49,10.57,CARDINAL,CARDINAL,spacy
2,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,131,141,Grant_Date,WORK_OF_ART,WORK_OF_ART,spacy
3,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,163,182,Abbott Laboratories,ORG,ORG,spacy
4,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,201,206,First,ORDINAL,ORDINAL,spacy
5,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,249,277,Performance Restricted Stock,ORG,ORG,spacy
6,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,283,288,Award,PERSON,PERSON,spacy
7,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,295,300,Award,WORK_OF_ART,WORK_OF_ART,spacy
8,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,307,320,NoShares12345,ORG,ORG,spacy
9,0000001800-24b040968ed13f8bd72ab3a7e7d0b619,346,368,the “Units”).The Award,ORG,ORG,spacy


## LexNLP (via the py3.8 container)

In [2]:
work = Path.cwd() / "ner_scratch"          # under the project -> Docker can mount it
work.mkdir(exist_ok=True)
df[[ID_COL, TEXT_COL]].to_parquet(work / "in.parquet")

subprocess.run(
    ["docker", "run", "--rm", "-v", f"{work}:/work", LEXNLP_IMAGE,
     "/work/in.parquet", "--output", "/work/out.parquet",
     "--id-col", ID_COL, "--text-col", TEXT_COL],
    check=True)

lexnlp_df = pd.read_parquet(work / "out.parquet")
print(len(lexnlp_df), "LexNLP candidates")
lexnlp_df.head(10)

Unable to find image 'contracts-lexnlp:latest' locally
docker: Error response from daemon: pull access denied for contracts-lexnlp, repository does not exist or may require 'docker login'

Run 'docker run --help' for more information


CalledProcessError: Command '['docker', 'run', '--rm', '-v', '/Users/matthiasuckert/RProjects/Projects/material-contracts/contracts-engine/ner_scratch:/work', 'contracts-lexnlp', '/work/in.parquet', '--output', '/work/out.parquet', '--id-col', 'DocID', '--text-col', 'TextRaw']' returned non-zero exit status 125.

## Check offsets + eyeball spans in context

In [ ]:
cand = pd.concat([spacy_df, lexnlp_df], ignore_index=True)
cand["Rehydrated"] = [text_by_id[d][s:e] for d, s, e in zip(cand.DocID, cand.Start, cand.Stop)]
cand["OffsetOk"] = cand.Rehydrated == cand.Span

# per-engine round-trip: text[Start:Stop] == Span ?
display(cand.groupby("Engine").OffsetOk.agg(["size", "sum", "mean"]))

# spans in context — does the offset land on a real entity?
def ctx(r):
    t = text_by_id[r.DocID]
    return t[max(r.Start - 25, 0):r.Stop + 25]

sample = cand.sample(min(15, len(cand)), random_state=1).copy()
sample["Context"] = sample.apply(ctx, axis=1)
sample[["Engine", "Label", "Span", "Context"]]